In [1]:
import os
from pathlib import Path
import json

from conch.open_clip_custom import create_model_from_pretrained
from conch.downstream.zeroshot_path import zero_shot_classifier, run_zeroshot

import torch
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from conch.open_clip_custom import tokenize, get_tokenizer
from tqdm import tqdm

/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/linus-dannull/anaconda3/envs/conch/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
model_cfg = 'conch_ViT-B-16'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
checkpoint_path = 'checkpoints/conch/pytorch_model.bin'
model, preprocess = create_model_from_pretrained(model_cfg, checkpoint_path, device=device)
_ = model.eval()

In [4]:
import json
import itertools

prompt_file = 'CONCH/prompts/crc100k_prompts_pruned.json'
with open(prompt_file) as f:
    data = json.load(f)["0"]

classnames = data["classnames"]
templates = data["templates"]

all_prompts = []

# expand into full prompt set
for class_key, synonyms in classnames.items():
    for syn in synonyms:
        for template in templates:
            if "CLASSNAME" in template:
                all_prompts.append(template.replace("CLASSNAME", syn))

In [5]:
tokenizer = get_tokenizer()

tokenized_prompts = tokenize(
    texts=all_prompts,
    tokenizer=tokenizer
).to(device)

with torch.no_grad():
    text_features = model.encode_text(tokenized_prompts)
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

In [6]:
from pathlib import Path
from PIL import Image

image_dir = Path("mhist_data/images")

results = []
k = 10 # the k-most similar prompts will be kept per image

for img_path in tqdm(list(image_dir.glob("*.png"))):
    image = Image.open(img_path).convert("RGB")
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        similarities = image_features @ text_features.T  # [1, num_prompts]

    topk_vals, topk_idxs = similarities[0].topk(k)

    top_prompts = [all_prompts[i] for i in topk_idxs.tolist()]
    top_scores = topk_vals.tolist()
    results.append({
        "image": img_path.name,
        "top_prompts": top_prompts,
        "top_scores": top_scores
    })

100%|██████████| 3152/3152 [12:54<00:00,  4.07it/s]


In [7]:
for i in range(len(results)):
    if 'adenocarcinoma' in results[i]['top_prompts'][0] and results[i]['top_scores'][0] > 0.7:
        print(results[i])
# that's a benign image btw lmao, according to mhist_data 0/7 thought that this was SSA (sessile serrated adenoma)

In [8]:
seven_outta_seven_ssa_imgs = ['dog', 'che', 'chr', 'cdf', 'cgg', 'dwa']
for i in range(len(results)):
    for img_str in seven_outta_seven_ssa_imgs:
        if img_str in results[i]['image']:
            print(results[i])

{'image': 'MHIST_che.png', 'top_prompts': ['normal colonic mucosa, H&E stain.', 'an H&E image showing uninvolved colon mucosa.', 'uninvolved colon mucosa, H&E.', 'an H&E stained image of normal colon mucosa.', 'an H&E image showing normal colonic mucosa.', 'an H&E stained image of normal colonic mucosa.', 'an H&E image showing normal colon mucosa.', 'uninvolved colon mucosa, H&E stain.', 'a photomicrograph showing normal colon mucosa.', 'an H&E image of normal colonic mucosa.'], 'top_scores': [0.6079355478286743, 0.602924108505249, 0.5973548889160156, 0.5933605432510376, 0.58467698097229, 0.5800677537918091, 0.5735667943954468, 0.5688080787658691, 0.5664718151092529, 0.5645726323127747]}
{'image': 'MHIST_dwa.png', 'top_prompts': ['uninvolved colon mucosa, H&E.', 'an H&E image of mucin pool.', 'an H&E stained image of mucin pool.', 'mucin.', 'mucin pool, H&E stain.', 'mucin pool, H&E.', 'uninvolved colon mucosa, H&E stain.', 'mucin, H&E stain.', 'normal colonic mucosa, H&E stain.', 'an 

In [ ]:
import csv
fieldnames = ["image", "top_prompts", "top_scores"]
with open("mhist_with_captions_pruned.csv", mode='w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames)
    writer.writeheader()
    writer.writerows(results)